# M3L3 E10 — Support bot baseline (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- medir un bot único antes del refactor.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — Baseline empresarial

AcmeOps tiene soporte HR, Tech y Billing. Antes de refactorizar, medimos un agente único con documentos mezclados.

> **Benchmark:** conjunto fijo de consultas para comparar versiones.


Creamos documentos mezclados y diez consultas: casos normales, mixtos y fuera de alcance.


In [ ]:
mixed_docs = [("hr","Vacaciones: 15 días."),("hr","Seguro desde el primer día."),("tech","VPN: reiniciar cliente y validar MFA."),("tech","Contraseña: portal de identidad."),("billing","Facturas antes del día 25."),("billing","Reembolsos con recibo y centro de costo.")]
benchmark_queries = [("vacaciones","hr"),("seguro","hr"),("vpn","tech"),("contraseña","tech"),("factura","billing"),("reembolso","billing"),("vacaciones y vpn","mixed"),("almuerzo","unknown"),("recibo","billing"),("notebook","tech")]


## Sección 2 — Agente único

`baseline_agent` predice un solo dominio y responde desde documentos mezclados. Esta limitación queda registrada para E11.


In [ ]:
def baseline_agent(query: str) -> dict:
    text = query.lower()
    if any(w in text for w in ["vacaciones", "seguro"]): pred = "hr"
    elif any(w in text for w in ["vpn", "contraseña", "notebook"]): pred = "tech"
    elif any(w in text for w in ["factura", "reembolso", "recibo"]): pred = "billing"
    else: pred = "unknown"
    context = [doc for domain, doc in mixed_docs if domain == pred]
    return {"predicted_domain": pred, "answer": " | ".join(context) or "No sé responder con confianza."}


## Sección 3 — Resultados

Calculamos accuracy simple. El caso mixto suele exponer el límite del baseline.


In [ ]:
results = []
for q, expected in benchmark_queries:
    r = baseline_agent(q); ok = r["predicted_domain"] == expected; results.append(ok)
    print(f"{expected:8} | pred={r['predicted_domain']:8} | ok={ok} | {q}")
print("Accuracy baseline:", sum(results), "/", len(results))


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    assert len(benchmark_queries) == 10
    assert isinstance(baseline_agent("vpn"), dict)
    print("Checks E10 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Medir un bot único antes del refactor.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
